In [1]:
import re
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredPowerPointLoader
from langchain_groq import ChatGroq
# RecursiveCharacterTextSplitter is a class that can be used to split text into smaller 
# chunks based on a specified separator. 
# It is useful for processing large texts and breaking them down into manageable pieces for 
# further analysis or processing.
import faiss
from langchain_community.vectorstores import FAISS
import numpy as np
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")


c:\Users\sabir_yetlff3\OneDrive\Desktop\Gen Ai\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import UnstructuredPowerPointLoader

data_dir = Path(r"C:\Users\sabir_yetlff3\OneDrive\Desktop\Gen Ai\PROJECT1\data")

documents = []
for pptx_file in data_dir.glob("*.pptx"):
    loader = UnstructuredPowerPointLoader(str(pptx_file))
    documents.extend(loader.load())

print(len(documents), "documents loaded")

3 documents loaded


In [3]:

### CHUNKING THE DOCUMENTS
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)



### EMBEDDING AND VECTOR STORE CREATION
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vector_store = FAISS.from_documents(docs, embedding_model)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})







In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

# def simple_rag(query):
    
#     # Step 1: Retrieve relevant docs
#     docs = retriever.invoke(query)   
    
#     # Step 2: Create context
#     context = "\n".join([doc.page_content for doc in docs])
    
#     # Step 3: Create prompt
#     prompt = f"""You are an HR Assistant.

#     Answer Only using the context below.
#     if answer is not present, say "I don't know".

#     Context: {context}

#     Question: {query}


#          """
    
#     # Step 4: Call LLM
#     response = llm.invoke(prompt)
    
#     return response.content


# query = "What is maternity leave policy?"

# answer = simple_rag(query)
# print(answer)

In [5]:
def classify_query(query):
    prompt = f"""
    classify query:
    - policy_question
    - general question

    Query : {query}
"""
    return llm.invoke(prompt).content

def hr_agent(query):
    intent = classify_query(query)
    
    if "policy" in intent:
        docs = retriever.invoke(query)
        context = "\n".join([d.page_content for d in docs])
    else:
        context = "No relevant documents found."

    prompt = f"""You are an HR Assistant.
    Context: {context}
    Question: {query} """

    return llm.invoke(prompt).content

query = "What is maternity leave policy?"
answer = hr_agent(query)

print(answer)

The maternity leave policy is as follows: 

1. For a woman having less than two surviving children, maternity leave with full salary will be allowed up to a maximum of 26 weeks, with not more than eight weeks preceding the date of expected delivery.

2. For a woman having two or more surviving children, maternity leave with full salary will be allowed up to a maximum of 12 weeks, with not more than six weeks preceding the date of expected delivery.

To avail of maternity leave, the employee must apply at least ten weeks before the expected date of delivery, along with a certificate from a registered gynecologist stating the expected date of delivery. The employee shall not take up any employment during the period of maternity leave. 

This benefit will not be applicable where provisions of the ESI Act are applicable to the employee. In such an event, the benefit will be given as per the provisions of the ESI Act, 1948.
